# Lead Times

This example shows how to use the `leadTime` parameter, available on every FINE component, to model
the delay between an investment decision and the physical availability of new capacity.

`leadTime` accepts the same three input shapes as other per-component, per-investment-period parameters
in FINE:

- a single **scalar** value -- applies uniformly to every location and every investment period
- a **Pandas Series** indexed by location -- varies by location, uniform across investment periods
- a **dict keyed by calendar year**, with one of the above as each value -- varies by both location
  *and* investment period

For a `leadTime` of `L` years, physical availability is delayed by `ceil(L / investmentPeriodInterval)`
investment periods relative to the investment/commissioning decision:

- **Investment costs (CAPEX)** are distributed across the *widened* `leadTime + economicLifetime`
  window, starting at the decision period -- a smaller, equal share is booked every period across this
  wider window (including during construction), rather than the narrower `economicLifetime`-only window
  used when `leadTime = 0`.
- **Fixed operating costs (OPEX)** are *shifted*, not widened: they only start being booked once the
  capacity is physically available, spread over the unchanged `technicalLifetime`.

`leadTime` defaults to `0`, which reproduces FINE's original behavior exactly (capacity available
immediately, no cost-timing change). See `Component`'s docstring for the full parameter description.

In [1]:
import fine as fn
import pandas as pd
import pyomo.environ as pyomo

## Model setup

Four otherwise-identical `Source` components are added to a two-location, three-investment-period energy
system, differing only in their `leadTime`:

| Component | `leadTime` | Shape |
|---|---|---|
| `source_no_lead` | `0` | scalar (baseline -- no delay, reproduces pre-`leadTime` behavior) |
| `source_scalar_lead` | `5` | scalar -- a uniform 5-year delay everywhere |
| `source_regional_lead` | `{"north": 0, "south": 10}` | per-location -- no delay in the north, a 10-year delay in the south |
| `source_regional_and_ip_lead` | varies by year *and* location (see below) | per-location-and-investment-period |

Each component's commissioning is fixed to 10 units in the first investment period (2020) only, and to 0
in every later period, so any differences below come purely from `leadTime`, not from different
investment decisions.

In [2]:
esM = fn.EnergySystemModel(
    locations={"north", "south"},
    commodities={"electricity"},
    commodityUnitsDict={"electricity": "GW_el"},
    numberOfTimeSteps=1,
    hoursPerTimeStep=1,
    startYear=2020,
    numberOfInvestmentPeriods=3,
    investmentPeriodInterval=5,
    costUnit="1e6 Euro",
    lengthUnit="km",
)

# 10 units commissioned in 2020 only, for every component.
commissioning_2020_only = {
    2020: pd.Series({"north": 10, "south": 10}),
    2025: pd.Series({"north": 0, "south": 0}),
    2030: pd.Series({"north": 0, "south": 0}),
}
operation_rate_max = pd.DataFrame([[1, 1]], index=[0], columns=["north", "south"])

common_kwargs = dict(
    commodity="electricity",
    hasCapacityVariable=True,
    operationRateMax=operation_rate_max,
    commissioningFix=commissioning_2020_only,
    investPerCapacity=100,
    opexPerCapacity=5,
    interestRate=0.05,
    economicLifetime=10,
)

esM.add(fn.Source(esM=esM, name="source_no_lead", leadTime=0, **common_kwargs))

esM.add(fn.Source(esM=esM, name="source_scalar_lead", leadTime=5, **common_kwargs))

esM.add(
    fn.Source(
        esM=esM,
        name="source_regional_lead",
        leadTime=pd.Series({"north": 0, "south": 10}),
        **common_kwargs,
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="source_regional_and_ip_lead",
        # north: lead time grows over the pathway (e.g. increasingly constrained grid
        # connection capacity); south: no delay at first, then a 5-year delay later.
        # Both per-location schedules are non-decreasing across investment periods,
        # which guarantees each decision period maps to a distinct availability
        # period (a decreasing schedule can make two decisions become available at
        # the same investment period -- FINE raises a clear error in that case).
        leadTime={
            2020: pd.Series({"north": 5, "south": 0}),
            2025: pd.Series({"north": 5, "south": 5}),
            2030: pd.Series({"north": 10, "south": 5}),
        },
        **common_kwargs,
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="sink",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=pd.DataFrame([[0, 0]], index=[0], columns=["north", "south"]),
    )
)

componentNames = [
    "source_no_lead",
    "source_scalar_lead",
    "source_regional_lead",
    "source_regional_and_ip_lead",
]

## Optimization

In [3]:
esM.optimize(
    timeSeriesAggregation=False,
    solver=fn.utils.ImplementedSolvers.STANDARD_SOLVER.value,
)

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter QCPDual to value 1


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))


...

Presolve: All rows and columns removed


Iteration    Objective       Primal Inf.    Dual Inf.      Time


       0    1.0938544e+04   0.000000e+00   0.000000e+00      0s


Solved in 0 iterations and 0.01 seconds (0.00 work units)


Optimal objective  1.093854401e+04


## How `leadTime` shifts capacity availability

All four components are commissioned (decided) identically: 10 units in 2020, nothing afterwards. The
`commis` column below is therefore identical across all four -- but the `cap` (physically available
capacity) column shows it becoming available at a different investment period for each, exactly matching
each component's `leadTime`:

- `source_no_lead`: available immediately in 2020.
- `source_scalar_lead`: available from 2025 (2020 + 5-year delay), everywhere.
- `source_regional_lead`: available from 2020 in the north (no delay), from 2030 in the south (10-year
  delay -> 2 investment periods).
- `source_regional_and_ip_lead`: available from 2025 in the north (5-year delay for the 2020 decision),
  from 2020 in the south (no delay for the 2020 decision).

In [4]:
def design_variable_table(esM, componentNames, locations=("north", "south")):
    abbrv = esM.componentModelingDict["SourceSinkModel"].abbrvName
    capVar = getattr(esM.pyM, f"cap_{abbrv}")
    commisVar = getattr(esM.pyM, f"commis_{abbrv}")
    rows = []
    for compName in componentNames:
        for loc in locations:
            for ip in esM.investmentPeriods:
                rows.append(
                    {
                        "component": compName,
                        "location": loc,
                        "year": esM.investmentPeriodNames[ip],
                        "commis": pyomo.value(commisVar[loc, compName, ip]),
                        "cap": pyomo.value(capVar[loc, compName, ip]),
                    }
                )
    return pd.DataFrame(rows).set_index(["component", "location", "year"])


design_variable_table(esM, componentNames)

commis   cap
component                   location year              
source_no_lead              north    2020    10.0  10.0
                                     2025     0.0  10.0
                                     2030     0.0   0.0
                            south    2020    10.0  10.0
                                     2025     0.0  10.0
                                     2030     0.0   0.0
source_scalar_lead          north    2020    10.0   0.0
                                     2025     0.0  10.0
                                     2030     0.0  10.0
                            south    2020    10.0   0.0
                                     2025     0.0  10.0
                                     2030     0.0  10.0
source_regional_lead        north    2020    10.0  10.0
                                     2025     0.0  10.0
                                     2030     0.0   0.0
                            south    2020    10.0   0.0
                                     2025     0.0   0.0
                                     2030     0.0  10.0
source_regional_and_ip_lead north    2020    10.0   0.0
                                     2025     0.0  10.0
                                     2030     0.0  10.0
                            south    2020    10.0  10.0
                                     2025     0.0  10.0
                                     2030     0.0   0.0

## How `leadTime` changes cost timing

- **`capexCap`** (annualized investment cost): for `leadTime = 0`, the full annualized cost is booked
  starting in the decision period, over the un-widened `economicLifetime` window (here: 2020-2025, 2
  investment periods). For `leadTime > 0`, the *same total* investment cost is instead spread over the
  wider `leadTime + economicLifetime` window as a smaller, equal share per period -- still starting at
  the *decision* period (2020), not the availability period.
- **`opexCap`** (fixed O&M cost): unlike CAPEX, this is *shifted*, not widened -- it stays zero until the
  capacity is physically available, then follows the same per-period amount `leadTime = 0` would have
  had, just starting later.

In [5]:
def cost_table(esM, componentNames, prop, locations=("north", "south")):
    rows = []
    for year in esM.investmentPeriodNames:
        summary = esM.getOptimizationSummary("SourceSinkModel", ip=year, outputLevel=0)
        for compName in componentNames:
            for loc in locations:
                value = summary.xs(
                    (compName, prop), level=("Component", "Property")
                )[loc].iloc[0]
                rows.append(
                    {
                        "component": compName,
                        "location": loc,
                        "year": year,
                        prop: value,
                    }
                )
    return (
        pd.DataFrame(rows)
        .set_index(["component", "location", "year"])[prop]
        .unstack("year")
    )


print("capexCap (annualized investment cost) per investment period:")
display(cost_table(esM, componentNames, "capexCap"))

capexCap (annualized investment cost) per investment period:


year                                        2020        2025       2030
component                   location                                   
source_no_lead              north     129.504575  129.504575   0.000000
                            south     129.504575  129.504575   0.000000
source_regional_and_ip_lead north      96.342288   96.342288  96.342288
                            south     129.504575  129.504575   0.000000
source_regional_lead        north     129.504575  129.504575   0.000000
                            south      80.242587   80.242587  80.242587
source_scalar_lead          north      96.342288   96.342288  96.342288
                            south      96.342288   96.342288  96.342288

In [6]:
print("opexCap (fixed O&M cost) per investment period:")
display(cost_table(esM, componentNames, "opexCap"))

opexCap (fixed O&M cost) per investment period:


year                                  2020  2025  2030
component                   location                  
source_no_lead              north     50.0  50.0   0.0
                            south     50.0  50.0   0.0
source_regional_and_ip_lead north      0.0  50.0  50.0
                            south     50.0  50.0   0.0
source_regional_lead        north     50.0  50.0   0.0
                            south      0.0   0.0  50.0
source_scalar_lead          north      0.0  50.0  50.0
                            south      0.0  50.0  50.0

## Summary

- `leadTime = 0` (the default) reproduces FINE's original behavior exactly: capacity is available
  immediately, and costs are booked exactly as without lead times.
- A scalar `leadTime` delays availability uniformly by the same number of investment periods everywhere.
- A per-location `leadTime` (Pandas Series) lets different locations have independent delays -- useful
  when, for example, permitting or grid-connection lead times differ by region.
- A per-investment-period-and-location `leadTime` (dict keyed by calendar year, valued by Series) allows
  the delay itself to change over the modeled pathway -- useful when lead times are expected to grow or
  shrink over time (e.g. due to changing permitting regimes or supply chain constraints).
- In every case, CAPEX widens to cover the full `leadTime + economicLifetime` window starting at the
  decision period, while OPEX simply shifts to start once the asset is physically available.